# Superoperator analysis from grid data

Analyse `data/grid/superoperator_N4_grid_<number>.npz`. Select the alpha and sigma values below; the corresponding plots only include those values.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
import numpy as np
import pandas as pd
PLOT_DIR = Path('docu/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## Load grid

Set the grid-file number here.

In [2]:
# npz-number
nbr = 1
path = Path(f"data/grid/superoperator_N4_grid_{nbr}.npz")
data = np.load(path)

print(f"Loaded: {path}")
print(f"Grid points: {len(data['h'])}")

Loaded: data/grid/superoperator_N4_grid_1.npz
Grid points: 600


## Alpha selection

The cell prints every alpha value in the file. Edit `alphas_to_plot` to choose the curves shown in the alpha plots; use `None` for all available values.

In [3]:
# (npz-number, alpha)
alpha_values = np.unique(data['alpha'])
print('Available alpha values:', alpha_values.tolist())

alphas_to_plot = [0.25, 0.5, 0.75, 1.0]  # set to None to include all alpha values
selected_alphas = alpha_values if alphas_to_plot is None else np.asarray(alphas_to_plot)
missing_alphas = np.setdiff1d(selected_alphas, alpha_values)
if len(missing_alphas):
    raise ValueError(f'alpha values not present in grid: {missing_alphas.tolist()}')
print('Alpha values included in plots:', selected_alphas.tolist())

Available alpha values: [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
Alpha values included in plots: [0.25, 0.5, 0.75, 1.0]


## Sigma selection

The cell prints every sigma value in the file. Edit `sigmas_to_plot` to choose the curves shown in the sigma plots; use `None` for all available values.

In [4]:
# (npz-number, sigma)
sigma_values = np.unique(data['sigma'])
print('Available sigma values:', sigma_values.tolist())

sigmas_to_plot = [0.5, 0.875, 1.25, 1.625, 2.0]  # set to None to include all sigma values
selected_sigmas = sigma_values if sigmas_to_plot is None else np.asarray(sigmas_to_plot)
missing_sigmas = np.setdiff1d(selected_sigmas, sigma_values)
if len(missing_sigmas):
    raise ValueError(f'sigma values not present in grid: {missing_sigmas.tolist()}')
print('Sigma values included in plots:', selected_sigmas.tolist())

Available sigma values: [0.5, 0.875, 1.25, 1.625, 2.0]
Sigma values included in plots: [0.5, 0.875, 1.25, 1.625, 2.0]


In [5]:
grid_data = pd.DataFrame({
    'h': data['h'],
    'alpha': data['alpha'],
    'sigma': data['sigma'],
    'omega_max': data['omega_max'],
    'Delta2': data['Delta2'],
    'trace_distance': data['trace_distance'],
    'get_mixingtime': data['get_mixingtime'],
})

print('h values:', np.unique(grid_data['h']).tolist())
print('omega_max values:', np.unique(grid_data['omega_max']).tolist())

h values: [0.2, 0.4, 0.6000000000000001, 0.8, 1.0, 1.2, 1.4000000000000001, 1.6, 1.8, 2.0]
omega_max values: [4.0, 8.0]


## Plots by alpha

Each line is one selected alpha value, at the chosen fixed sigma and `omega_max`.

In [6]:
from matplotlib.ticker import NullFormatter

sigma_for_alpha_plot = 2.0
omega_values_to_plot = [4.0, 8.0]

plt.style.use("default")

alpha_plot_data = grid_data[
    grid_data['alpha'].isin(selected_alphas)
    & np.isclose(grid_data['sigma'], sigma_for_alpha_plot)
].copy()
missing_omegas = np.setdiff1d(omega_values_to_plot, alpha_plot_data['omega_max'].unique())
if alpha_plot_data.empty or len(missing_omegas):
    raise ValueError(f'No data for alpha plot omega_max values: {missing_omegas.tolist()}')

# Use raw strings (r'...') for titles containing LaTex, e.g. r'Spectral gap $\Delta_2$'.
alpha_plot_titles = {
    'Delta2': r'Spectral separation $\Delta_2$',
    'trace_distance': r'Distance of fixed point to Gibbs state',
    'get_mixingtime': r'Convergence to fixed point',
}

iteration_values = alpha_plot_data['get_mixingtime'].to_numpy()
positive_iterations = iteration_values[np.isfinite(iteration_values) & (iteration_values > 0)]
if len(positive_iterations) == 0:
    raise ValueError('No positive finite mixing-time values are available for plotting.')
log_lower = np.log10(positive_iterations.min()) - 0.1
log_upper = np.log10(positive_iterations.max()) + 0.1
iteration_ylim = (10**log_lower, 10**log_upper)
iteration_ticks = [
    mantissa * 10**exponent
    for exponent in range(int(np.floor(log_lower)) - 1, int(np.ceil(log_upper)) + 2)
    for mantissa in (1, 2, 5)
    if iteration_ylim[0] <= mantissa * 10**exponent <= iteration_ylim[1]
]

for column, ylabel in zip(
    ['Delta2', 'trace_distance', 'get_mixingtime'],
    [r'$\Delta_2$', r'$\frac{1}{2}\|\rho^* - \rho_\beta\|_1$', 'iterations to fixed point'],
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
    for ax, omega_max in zip(axes, omega_values_to_plot):
        plot_slice = alpha_plot_data[np.isclose(alpha_plot_data['omega_max'], omega_max)]
        for alpha in selected_alphas:
            line_data = plot_slice[np.isclose(plot_slice['alpha'], alpha)].sort_values('h')
            ax.plot(line_data['h'], line_data[column], marker='o', label=rf'$\alpha = {alpha:g}$')
        ax.set_xlabel(r'$h/J$')
        ax.set_ylabel(ylabel)
        if column == 'get_mixingtime':
            ax.set_yscale('log')
            ax.set_ylim(*iteration_ylim)
            ax.set_yticks(iteration_ticks)
            ax.set_yticklabels([f'{tick:g}' for tick in iteration_ticks])
            ax.yaxis.set_minor_formatter(NullFormatter())
        ax.set_title(f'{alpha_plot_titles[column]}\n' + rf'$\sigma = {sigma_for_alpha_plot:g}$, $\omega_{{max}} = {omega_max:g}$')
        ax.legend(title=r'$\alpha$')
        ax.grid(True)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'grid_alpha_{column}.png', dpi=300, bbox_inches='tight')
    plt.show()

/var/folders/58/t80f2npn6t1bn5v6frsvtrnw0000gn/T/ipykernel_31402/2646728921.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/58/t80f2npn6t1bn5v6frsvtrnw0000gn/T/ipykernel_31402/2646728921.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Log-linear fits in two h/J ranges: log(iterations) = slope * h + intercept
r2_rows = []
h_ranges = {
    r'all $h/J$': lambda h_values: np.ones_like(h_values, dtype=bool),
    r'$h/J < 1$': lambda h_values: h_values < 1,
    r'$h/J \geq 1$': lambda h_values: h_values >= 1,
}
for omega_max in omega_values_to_plot:
    omega_data = alpha_plot_data[np.isclose(alpha_plot_data['omega_max'], omega_max)]
    for alpha in selected_alphas:
        line_data = omega_data[np.isclose(omega_data['alpha'], alpha)].sort_values('h')
        x = line_data['h'].to_numpy()
        iterations = line_data['get_mixingtime'].to_numpy()
        fit_mask = np.isfinite(x) & np.isfinite(iterations) & (iterations > 0)
        for h_range, range_mask in h_ranges.items():
            range_fit_mask = fit_mask & range_mask(x)
            if np.count_nonzero(range_fit_mask) < 2:
                slope, intercept, r_squared = np.nan, np.nan, np.nan
            else:
                slope, intercept = np.polyfit(x[range_fit_mask], np.log(iterations[range_fit_mask]), deg=1)
                fitted_log_iterations = slope * x[range_fit_mask] + intercept
                residual_sum_squares = np.sum((np.log(iterations[range_fit_mask]) - fitted_log_iterations) ** 2)
                total_sum_squares = np.sum((np.log(iterations[range_fit_mask]) - np.mean(np.log(iterations[range_fit_mask]))) ** 2)
                r_squared = np.nan if np.isclose(total_sum_squares, 0) else 1 - residual_sum_squares / total_sum_squares
            r2_rows.append({'alpha': alpha, 'omega_max': omega_max, 'h_range': h_range, 'slope': slope, 'intercept': intercept, 'r_squared': r_squared})

alpha_iteration_fit = pd.DataFrame(r2_rows)
# display(alpha_iteration_fit)

# This may be a raw LaTex string, e.g. r'Log-linear fit quality $R^2$'.
r2_plot_title = r'Log-linear fit quality $R^2$'
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, omega_max in zip(axes, omega_values_to_plot):
    r2_data = alpha_iteration_fit[np.isclose(alpha_iteration_fit['omega_max'], omega_max)]
    bar_width = 0.07
    for h_range, offset in [(r'$h/J < 1$', -bar_width / 2), (r'$h/J \geq 1$', bar_width / 2)]:
        range_data = r2_data[r2_data['h_range'] == h_range].sort_values('alpha')
        ax.bar(range_data['alpha'] + offset, range_data['r_squared'], width=bar_width, label=h_range)
    whole_interval_data = r2_data[r2_data['h_range'] == r'all $h/J$'].sort_values('alpha')
    ax.plot(whole_interval_data['alpha'], whole_interval_data['r_squared'], 'kd', markersize=9, label=r'all $h/J$', zorder=3)
    ax.set_xlabel(r'$\alpha$')
    ax.set_ylabel(r'$R^2$')
    ax.set_xticks(selected_alphas)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f'{r2_plot_title}\n' + rf'$\sigma = {sigma_for_alpha_plot:g}$, $\omega_{{max}} = {omega_max:g}$')
    ax.legend()
    ax.grid(True)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'grid_alpha_iteration_fit_r2.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/58/t80f2npn6t1bn5v6frsvtrnw0000gn/T/ipykernel_31402/110039615.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Plots by sigma

Each line is one selected sigma value, at the chosen fixed alpha and `omega_max`.

In [8]:
from matplotlib.ticker import NullFormatter

alpha_for_sigma_plot = 0.25
omega_values_to_plot = [4.0, 8.0]

sigma_plot_data = grid_data[
    grid_data['sigma'].isin(selected_sigmas)
    & np.isclose(grid_data['alpha'], alpha_for_sigma_plot)
].copy()
missing_omegas = np.setdiff1d(omega_values_to_plot, sigma_plot_data['omega_max'].unique())
if sigma_plot_data.empty or len(missing_omegas):
    raise ValueError(f'No data for sigma plot omega_max values: {missing_omegas.tolist()}')

# Use raw strings (r'...') for titles containing LaTex, e.g. r'Spectral gap $\Delta_2$'.
sigma_plot_titles = {
    'Delta2': r'Spectral separation $\Delta_2$',
    'trace_distance': r'Distance of fixed point to Gibbs state',
    'get_mixingtime': r'Convergence to fixed point',
}

for column, ylabel in zip(
    ['Delta2', 'trace_distance', 'get_mixingtime'],
    [r'$\Delta_2$', r'$\frac{1}{2}\|\rho^* - \rho_\beta\|_1$', 'iterations to fixed point'],
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=column != 'get_mixingtime')
    for ax, omega_max in zip(axes, omega_values_to_plot):
        plot_slice = sigma_plot_data[np.isclose(sigma_plot_data['omega_max'], omega_max)]
        for sigma in selected_sigmas:
            line_data = plot_slice[np.isclose(plot_slice['sigma'], sigma)].sort_values('h')
            ax.plot(line_data['h'], line_data[column], marker='o', label=rf'$\sigma = {sigma:g}$')
        ax.set_xlabel(r'$h/J$')
        ax.set_ylabel(ylabel)
        if column == 'get_mixingtime':
            iteration_values = plot_slice['get_mixingtime'].to_numpy()
            positive_iterations = iteration_values[np.isfinite(iteration_values) & (iteration_values > 0)]
            tick_candidates = [
                mantissa * 10**exponent
                for exponent in range(int(np.floor(np.log10(positive_iterations.min()))) - 1, int(np.ceil(np.log10(positive_iterations.max()))) + 2)
                for mantissa in (1, 2, 5)
            ]
            lower_tick = max(tick for tick in tick_candidates if tick <= positive_iterations.min())
            upper_tick = min(tick for tick in tick_candidates if tick >= positive_iterations.max())
            iteration_ylim = (lower_tick, upper_tick)
            iteration_ticks = [tick for tick in tick_candidates if lower_tick <= tick <= upper_tick]
            ax.set_yscale('log')
            ax.set_ylim(*iteration_ylim)
            ax.set_yticks(iteration_ticks)
            ax.set_yticklabels([f'{tick:g}' for tick in iteration_ticks])
            ax.yaxis.set_minor_formatter(NullFormatter())
        ax.set_title(f'{sigma_plot_titles[column]}\n' + rf'$\alpha = {alpha_for_sigma_plot:g}$, $\omega_{{max}} = {omega_max:g}$')
        ax.legend(title=r'$\sigma$')
        ax.grid(True)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'grid_sigma_{column}.png', dpi=300, bbox_inches='tight')
    plt.show()

/var/folders/58/t80f2npn6t1bn5v6frsvtrnw0000gn/T/ipykernel_31402/1858013582.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/58/t80f2npn6t1bn5v6frsvtrnw0000gn/T/ipykernel_31402/1858013582.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
